In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@databricksuk2025.dfs.core.windows.net/customers")

# Remove Column
df = df.drop('_rescued_data')

df.display()

In [0]:
# SPLIT FUNCTION

df = df.withColumn("domains",split(col('email'),'@')[1])
df.display()

In [0]:
# Aggregating

df.groupBy("domains")\
    .agg(count("customer_id")\
    .alias("total_customers"))\
    .sort("total_customers",ascending=False)\
    .display()

In [0]:
df_gmail = df.filter(col('domains')=='gmail.com').display()

In [0]:
# CONCATE FUNCTION

df = df.withColumn("full_name",concat(col('first_name'),lit(' '),col('last_name')))
df = df.drop('first_name','last_name')
df.display()

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@databricksuk2025.dfs.core.windows.net/customers")

In [0]:
%sql
Create table if not exists databricks_cata.silver.customers_silver 
using DELTA
Location 'abfss://silver@databricksuk2025.dfs.core.windows.net/customers'




In [0]:
%sql

select * from databricks_cata.silver.customers_silver